In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import h5py
from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=3)

In [ ]:
####################################
#DATA LOADING

In [ ]:
# SUBSETTING SBF and COLDPOOL PARCELS

#getting convergence xmax
def Get_AvgConvergence(t):

    timeString = ModelData.timeStrings[t]
    outputDataDirectory=os.path.normpath(os.path.join(DataManager.outputDataDirectory,"..","Eulerian_CLTracking"))
    Dictionary = TrackingAlgorithms_DataLoading_Class.LoadData(ModelData, DataManager, timeString,
                     dataName="Eulerian_CLTracking",outputDataDirectory=outputDataDirectory,printstatement=False)
    avgConvergence = Dictionary["avgConvergence"]
    return avgConvergence

def find_SBF_xmaxs():
    xmaxs=[]
    for t in tqdm(range(ModelData.Ntime)):
        if t == 0:
            xmaxs.append(np.nan)
        else:
            avgConvergence = Get_AvgConvergence(t)
            avgConvergence_max=np.nanmax(avgConvergence)
            xmax = np.where(avgConvergence==avgConvergence_max)[0][0]
            xmaxs.append(xmax)
    return xmaxs

def Get_SBF_xmaxs_FilePath(ModelData, DataManager):
    fileName = (
        f"SBF_xmaxs_{ModelData.res}_{ModelData.t_res}_"
        f"{ModelData.Ntime}nt.pkl"
    )
    return os.path.join(DataManager.outputDataDirectory, fileName)

def LoadOrRun_find_SBF_xmaxs(ModelData, DataManager, overwrite=False):
    filePath = Get_SBF_xmaxs_FilePath(ModelData, DataManager)
    
    if os.path.exists(filePath) and not overwrite:
        print(f"reading from {filePath}")
        with open(filePath, "rb") as f:
            xmaxs = pickle.load(f)
        return xmaxs

    xmaxs = find_SBF_xmaxs()

    os.makedirs(os.path.dirname(filePath), exist_ok=True)
    with open(filePath, "wb") as f:
        pickle.dump(xmaxs, f)
    print(f"saved to {filePath}")
    return xmaxs

DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")
try:
    xmaxs = LoadOrRun_find_SBF_xmaxs(ModelData, DataManager)
except:
    xmaxs = [ModelData.Nxh]*ModelData.Ntime

In [ ]:
####################################
#PLOTTING FUNCTIONS
if ModelData.Nzh==34:
    allData=ModelData.OpenData()

In [ ]:
def CalculateCloudTopArray():
    
    cloud_top_arr = np.full((ModelData.Ntime, len(xh)), np.nan)
    
    times=np.arange(0, ModelData.Ntime)
    for t in tqdm(times):
        if ModelData.Nzh==34:
            data=allData.isel(time=t) 
        else:
            data=ModelData.OpenData_SingleTime(t=t).isel(time=0)
        qcqi = data['qc']+data['qi']
        
        a = qcqi.data
        a[a < 1e-6] = np.nan
        
        valid_mask = ~np.isnan(a)  # shape (z, x)
        top_idx = np.argmax(valid_mask[::-1, :], axis=0)
        top_idx = (valid_mask.shape[0] - 1) - top_idx
        cloud_top = np.where(valid_mask.any(axis=0), ModelData.zh[top_idx], np.nan)
        cloud_top_arr[t, :] = np.nanmean(cloud_top,axis=0)
    return cloud_top_arr

xh = ModelData.xh-ModelData.xh[0]
        
def PlotCloudTopArray_contour(cloud_top_arr):
    fig, ax = plt.subplots()
    time=np.arange(ModelData.Ntime)*ModelData.dt/3600+6
    plt.contourf(xh,time,cloud_top_arr);plt.colorbar(label='CT',pad=0.01)
    plt.contour(xh,time,cloud_top_arr>=6,colors='red')
    plt.contour(xh,time,cloud_top_arr<=4,colors='white')
    plt.plot(xmaxs, time, color='black', linewidth=2, label='Sea breeze front')
    
    plt.xlabel('x (km)'); plt.ylabel('Local Time')
    plt.xlim(left=xh[-1]*0.25,right=xh[-1]);plt.ylim(bottom=10)
    plt.title('Plot of Average CT using qcqi >= 1e-6 threshold')
    
    #legend
    legend_elements = [Line2D([0],[0], color='red', label='CT >= 6km'),
                       Line2D([0],[0], color='white', label='CT <= 4km'),
                       Line2D([0],[0], color='black', label='SBF x + 10')]
    plt.legend(handles=legend_elements, loc='lower right')

    return fig,ax

def PlotCloudTopArray_line(cloud_top_arr):
    import matplotlib
    fig, ax = plt.subplots(figsize=(12, 5))
    times = np.arange(ModelData.Ntime)
    time = np.arange(ModelData.Ntime)*ModelData.dt/3600+6
    n_levels = 10
    cmap = plt.cm.viridis_r
    bounds = np.linspace(times.min(), times.max(), n_levels + 1)
    norm = matplotlib.colors.BoundaryNorm(bounds, cmap.N)
    

    step = int(np.ceil(ModelData.Ntime/n_levels)); last_band = -1
    for t in tqdm(times):
        ax.plot(xh, cloud_top_arr[t, :], color=cmap(norm(t)), alpha=0.7)
        if (t // step) != last_band:
            ax.axvline(xmaxs[t]+10, color=cmap(norm(t)), alpha=0.7, linewidth=1.5); last_band = (t // step)
    
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    cb = plt.colorbar(sm, ax=ax, label='Local Time', ticks=bounds, pad=0.01)
    cb.set_ticklabels([f'{time[int(b)]:.1f}' for b in bounds])
    ax.set_xlabel('x'); ax.set_ylim(bottom=0)
    plt.xlim(left=xh[-1]*0.25,right=xh[-1])
    ax.set_ylabel('Cloud top height')
    ax.axhline(6, color='red', zorder=10)
    ax.axhline(4, color='red', zorder=10)
    ax.axvline(xh[-1]-50, color='red', zorder=10)
    ax.set_title('Plot of Average CT using qcqi >= 1e-6 threshold')
    plt.tight_layout()
    return fig,ax

In [ ]:
####################################
#PLOTTING

In [ ]:
cloud_top_arr = CalculateCloudTopArray()
[fig,ax] = PlotCloudTopArray_contour(cloud_top_arr)
[fig,ax] = PlotCloudTopArray_line(cloud_top_arr)

/tmp/ipykernel_2609576/3142624179.py:20: RuntimeWarning: Mean of empty slice
  cloud_top_arr[t, :] = np.nanmean(cloud_top,axis=0)
  7%|▋         | 15/221 [00:07<01:52,  1.83it/s]/tmp/ipykernel_2609576/3142624179.py:20: RuntimeWarning: Mean of empty slice
  cloud_top_arr[t, :] = np.nanmean(cloud_top,axis=0)
  7%|▋         | 16/221 [00:08<01:45,  1.94it/s]/tmp/ipykernel_2609576/3142624179.py:20: RuntimeWarning: Mean of empty slice
  cloud_top_arr[t, :] = np.nanmean(cloud_top,axis=0)
  8%|▊         | 17/221 [00:08<01:42,  1.99it/s]/tmp/ipykernel_2609576/3142624179.py:20: RuntimeWarning: Mean of empty slice
  cloud_top_arr[t, :] = np.nanmean(cloud_top,axis=0)
  8%|▊         | 18/221 [00:09<01:38,  2.05it/s]/tmp/ipykernel_2609576/3142624179.py:20: RuntimeWarning: Mean of empty slice
  cloud_top_arr[t, :] = np.nanmean(cloud_top,axis=0)
  9%|▊         | 19/221 [00:09<01:52,  1.80it/s]/tmp/ipykernel_2609576/3142624179.py:20: RuntimeWarning: Mean of empty slice
  cloud_top_arr[t, :] = np.nanmea